# Figure 2 — single dye molecules on the Ximea camera

**Makes up:** Figure 2 b and c (a field of view of single ATTO 565 / ATTO 633 molecules with
single-step photobleaching traces and PSF zoom-ins) and e (ternary plot of eight dyes' relative
pixel QE, 1,000 molecules each). SI: `Colour_vs_Prediction` (measured vs predicted relative pixel
QE of all twelve dyes) and `Single_Step_Photobleaching` (twelve photobleaching traces per dye).

**Not here:** Figure 2a is a schematic. Figure 2d (TetraSpeck bead precision) is made by
`Figure_2d_TetraspeckPrecision.ipynb`.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/Single_Dye_Experiments/<dye>/`:
twelve dyes, each a 4 × 4 grid of positions (one TIFF of up to 1,250 frames at 100 ms, 1.4–2 GB,
plus one MicroManager metadata file each). The deposition holds raw data only, so this notebook fits it.

**Method:** fit each position (`fit_SM_data`, 12 px ROI, PSF wavelength = the dye's emission peak) →
HDBSCAN clustering of each dye's localisations into molecules (≥ 10 localisations, as in the
paper's Methods). Photobleaching: the raw 12 × 12 px photoelectron trace around each molecule must
show exactly one downward step (likelihood-ratio change-point detector). Shared code:
`_multifov.py` (download / fit / cluster) and `_single_dyes.py` (dyes, filters, bleaching screen).

## FAST_MODE

`FAST_MODE = True` (default) uses one position per dye (12 TIFFs of 1.4–2 GB, ~20 GB): ATTO 633's is
the position shown in the paper (`Pos002_003`), the others the first. Fitting takes ~2 min per
position; the download from EBI usually takes far longer (~15 min per GB-scale file at the speeds
seen). Every panel is made, from ~1/16 of the molecules. `FAST_MODE = False` uses all 192
positions (~300 GB; run in `tmux`/`screen`). Panels b and c then show each dye's best position.

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mpltern  # noqa: F401  (registers the "ternary" projection)
from scipy import ndimage

# _biostudies_download.py, _multifov.py and _single_dyes.py live beside this notebook.
sys.path.insert(0, str(Path.cwd().resolve()))
import _multifov as mf
import _single_dyes as sd

from pyS3M import IOFunctions
from pyS3M import HelperFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.StepDetector import StepDetector

IO = IOFunctions.IO_Functions()
H_F = HelperFunctions.Helper_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: one position per dye (~20 GB). False: all 192 positions (~300 GB, the published data).
REFIT = False      # True: fit again even if fits with the same files and settings exist.
FAST_POSITIONS = {"ATTO633": "Pos002_003"}   # FAST_MODE: the position Fig. 2c shows; others take the first

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_2_SingleDyes"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each dye gets its own sub-folder,
# separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download and fit every dye

Per dye: download from BioStudies (with each position's metadata: the camera ROI the fit needs),
then fit each position separately. The PSF model's wavelength is the dye's emission peak.
Re-running is safe: complete downloads are skipped, and fits are reused unless the files or
settings change (or `REFIT = True`).

In [ ]:
dye_files, exposure_s = {}, {}
for dye, (spectrum_name, folder, _, _) in sd.DYES.items():
    fit_params = dict(sd.FIT_PARAMS, peak_wavelength=sd.emission_peak_um(S_F, spectrum_name, wavelength))
    print(f"\n=== {dye} (PSF wavelength {fit_params['peak_wavelength']} um)")
    dye_files[dye] = mf.prepare_acquisition(
        f"{sd.DYE_ROOT}/{folder}", DATA_ROOT / dye, FAST_MODE, SupRes_F, calib, smoothing_function, IO,
        fit_params, position=FAST_POSITIONS.get(dye), refit=REFIT,
    )
    exposure_s[dye] = IO.metadata_reader_imageJ(mf.metadata_for(mf.tif_for(dye_files[dye][0])),
                                                return_exposure=True)[4] / 1000

for dye, files in dye_files.items():
    print(f"{dye}: {len(files)} position(s), {exposure_s[dye] * 1000:.0f} ms frames")

---
## 3 — Cluster each dye's localisations into molecules

HDBSCAN, min. cluster size 10 localisations, after quality cuts (chi² < 2, localisation error
< 1 px, 100–10⁶ photons). A molecule's colour (`A_B`, `A_G`, `A_R`: relative signal on pixels 1,
2 and 3) is the inverse-variance weighted mean over its localisations; its photons are their sum.

In [ ]:
sm_db, sf_db = {}, {}
for dye, files in dye_files.items():
    sm_db[dye], sf_db[dye] = mf.cluster_sample(SM_E, files, DATA_ROOT / dye, dye, sd.CLUSTER_PARAMS)
    print(f"{dye}: {len(sm_db[dye]):,} molecules, median {sm_db[dye]['photons'].median():,.0f} photons")

---
## 4 — Figure 2e: eight dyes on a ternary plot

1,000 molecules of each dye with 2 × 10⁴–2 × 10⁵ photons, drawn at random. Full triangle, and a
zoom on the red dyes. Axes: top = pixel 3 (`A_R`), left = pixel 2 (`A_G`), right = pixel 1
(`A_B`). (The analysis notebook labelled the top and right axes the other way round: pixel 1 and
pixel 2.) ATTO 700 is drawn in near-white, as published, so it only shows on a dark background.

In [ ]:
PHOTON_RANGE = (2e4, 2e5)
N_SHOWN = 1000
rng = np.random.default_rng(0)

ternary_dyes = [d for d, (_, _, _, colour) in sd.DYES.items() if colour is not None]
shown = {}
for dye in ternary_dyes:
    db = sm_db[dye][sm_db[dye]["photons"].between(*PHOTON_RANGE)]
    shown[dye] = db.iloc[rng.choice(len(db), min(N_SHOWN, len(db)), replace=False)]
    print(f"{dye}: {len(db):,} of {len(sm_db[dye]):,} molecules in range, {len(shown[dye])} shown "
          f"(median {db['photons'].median():,.0f} photons)")


def ternary_panel(limits, alpha, size_in, name):
    fig = plt.figure(figsize=(size_in, size_in))
    ax = fig.add_subplot(projection="ternary", ternary_sum=1)
    for dye in ternary_dyes:
        db = shown[dye]
        ax.scatter(db["A_R"], db["A_G"], db["A_B"], color=sd.DYES[dye][3], marker=".", s=1,
                   alpha=alpha, rasterized=True, label=sd.DYES[dye][0])
    ax.set_ternary_lim(*limits)
    ax.set_tlabel("pixel 3 QE")
    ax.set_llabel("pixel 2 QE")
    ax.set_rlabel("pixel 1 QE")
    ax.grid(lw=0.25, alpha=0.25, ls="--", which="both", axis="both", color="gray")
    plt.savefig(FIG_DIR / name, dpi=600, format="svg")
    plt.show()


ternary_panel((0, 1, 0, 1, 0, 1), alpha=0.05, size_in=2.5, name="ternary_rasterized.svg")
# red zoom: pixel 3 0.6-0.8, pixel 2 0.2-0.4, pixel 1 0-0.2
ternary_panel((0.6, 0.8, 0.2, 0.4, 0.0, 0.2), alpha=0.4, size_in=3.33, name="red_zoom_tern_large.svg")

---
## 5 — SI: measured vs predicted relative pixel QE, all twelve dyes

Histograms of every molecule's `A_B`, `A_G`, `A_R`; dashed lines: the prediction from the dye's
emission spectrum through that acquisition's filters and the objective, integrated against
each pixel's QE.

In [ ]:
fig, axs = plotter.two_column_plot(nrows=4, ncols=3, height=8)
bins = np.linspace(0, 1, 101)
for ax, dye in zip(axs.ravel(), sd.DYES):
    db = sm_db[dye]
    predicted = sd.predicted_fingerprint(S_F, dye, wavelength, pixel_QYs)   # B, G, R
    for column, colour, label, p in [("A_B", "blue", "pixel 1 QE", predicted[0]),
                                     ("A_G", "darkgreen", "pixel 2 QE", predicted[1]),
                                     ("A_R", "red", "pixel 3 QE", predicted[2])]:
        plotter.histogram_plot(ax, db[column], bins=bins, color=colour, alpha=0.5, xlim=(0, 1),
                               label=label, xlabel="relative pixel QE", ylabel="")
        ax.axvline(p, color=colour, ls="--", label=f"predicted {label}")
    ax.set_title(sd.DYES[dye][0])
axs.ravel()[0].legend(fontsize=5)
plt.savefig(FIG_DIR / "colour_vs_prediction.svg", dpi=600, format="svg")
plt.show()

---
## 6 — Screen every dye for single-step photobleaching

For each molecule seen in at least 25 frames, its 12 × 12 px photoelectron trace (first detection
to 50 frames after the last) must show exactly one downward step. Candidates are ranked by
on/off contrast. This reads each position's TIFF once (memory-mapped), roughly 20 s per position.

In [ ]:
screen = sd.BleachScreen(StepDetector(win_size=sd.STEP_WIN_SIZE, alpha=sd.STEP_ALPHA, estimator="gaussian"),
                         IO, H_F, calib)
bleachers = {}
for dye in sd.DYES:
    bleachers[dye] = screen.screen(sf_db[dye], dye_files[dye])
    print(f"{dye}: {len(bleachers[dye]):,} single-step bleachers")

---
## 7 — SI: twelve single-step photobleaching traces per dye

The twelve highest-contrast traces of each dye (baseline-subtracted), with the step's segment means.

In [ ]:
N_ROWS, N_COLS = 3, 4
for dye in sd.DYES:
    fig, axs = plotter.two_column_plot(nrows=N_ROWS, ncols=N_COLS)
    axs = np.atleast_1d(axs).ravel()
    for i, ax in enumerate(axs):
        if i >= len(bleachers[dye]):
            ax.set_visible(False)
            continue
        cand = bleachers[dye][i]
        trace = cand["trace"] - cand["trace"].min()
        frames = np.arange(len(trace))
        ax.plot(frames, trace, color="black", lw=0.5, alpha=0.9, zorder=1)
        ax.plot(frames, screen.detector.segment_means(trace, cand["cps"]), color="#d40000", lw=1.1, zorder=2)
        ax.axvline(cand["cp"], color="#d40000", lw=0.8, ls="--", alpha=0.6, zorder=3)
        ax.set_xlim(frames[0], frames[-1])
        ax.set_ylim(bottom=0)
        ax.tick_params(labelsize=7, length=2, pad=1)
        if i % N_COLS == 0:
            ax.set_ylabel("intensity / pe", fontsize=8)
        if i // N_COLS == N_ROWS - 1:
            ax.set_xlabel("frame", fontsize=8)
    fig.savefig(FIG_DIR / f"{dye}_SI_Single_Dye_Photobleaching.svg", dpi=300, format="svg", bbox_inches="tight")
    plt.show()

---
## 8 — Figure 2b, c: ATTO 565 and ATTO 633 fields of view

The position with the four best, well-separated (≥ 10 px) single-step bleachers that are not
hidden by the trace inset. Left: maximum
projection (photoelectrons) over their on-times, each boxed, with their traces inset. Right:
12 × 12 px zoom-ins of the four PSFs (maximum projection over each molecule's own on-time). For
ATTO 633 each zoom is centred on the green-channel intensity centroid; ATTO 565's positions
needed no re-centring.

In [ ]:
BOX_COLOURS = ["#e6194b", "#3cb44b", "#4363d8", "#ff9f00"]
N_TRACES = 4
HALF = sd.ROI_SIZE // 2   # trace ROI half-width
ZOOM_HALF = 11            # window searched for the zoom-in centre
FINAL_HALF = 6            # zoom-in half-width (12 x 12 px)
FOV_IN, INSET_W, INSET_H, MARGIN, GAP = 2.5, 1.5, 1.5, 0.05, 0.04   # inches


def visible(cand, width, height):
    # False if the molecule's box would sit under the trace inset (image drawn with equal aspect,
    # centred in a FOV_IN x FOV_IN figure; inset in its bottom-right corner).
    scale = min(FOV_IN / width, FOV_IN / height)            # inches per camera px
    left, bottom = (FOV_IN - width * scale) / 2, (FOV_IN - height * scale) / 2
    x_in = left + cand["xc"] * scale                        # rows run top to bottom (origin="upper")
    y_in = bottom + (height - cand["yc"]) * scale
    pad = HALF * scale
    return not (x_in + pad > FOV_IN - INSET_W - MARGIN and y_in - pad < MARGIN + INSET_H)


def zoom_crop(window, recentre):
    if recentre:
        green = sCMOS.bayer_demosaic_stack(window)[0][:, :, 1]
        cy, cx = (int(round(v)) for v in ndimage.center_of_mass(green))
    else:
        cy, cx = window.shape[0] // 2, window.shape[1] // 2
    y0 = min(max(0, cy - FINAL_HALF), window.shape[0] - 2 * FINAL_HALF)
    x0 = min(max(0, cx - FINAL_HALF), window.shape[1] - 2 * FINAL_HALF)
    return window[y0:y0 + 2 * FINAL_HALF, x0:x0 + 2 * FINAL_HALF]


for dye in ["ATTO565", "ATTO633"]:
    by_fov = {}
    for cand in bleachers[dye]:
        fov = cand["fov_index"]
        _, _, w, h = IO.metadata_reader_imageJ(mf.metadata_for(mf.tif_for(dye_files[dye][fov])))
        if visible(cand, w, h):
            by_fov.setdefault(fov, []).append(cand)
    eligible = {fov: sd.pick_spread(c, N_TRACES) for fov, c in by_fov.items()}
    eligible = {fov: c for fov, c in eligible.items() if len(c) == N_TRACES}
    if not eligible:
        print(f"{dye}: no position with {N_TRACES} well-separated bleachers")
        continue
    best_fov = max(eligible, key=lambda fov: sum(c["snr_ratio"] for c in eligible[fov]))
    top = eligible[best_fov]
    tif = mf.tif_for(dye_files[dye][best_fov])
    print(f"{dye}: {tif.name}, contrasts {[round(c['snr_ratio'], 1) for c in top]}")

    stack = sd.open_stack(tif)
    gain, offset, rqe = screen._maps_for(tif)
    n_frames, height, width = stack.shape
    f_lo, f_hi = min(c["f0"] for c in top), max(c["f0"] + c["cp"] for c in top)
    fov_img = screen.to_pe(stack[f_lo:f_hi].max(axis=0), gain, offset, rqe, 0, height, 0, width)
    for cand in top:
        xpx, ypx = int(round(cand["xc"])), int(round(cand["yc"]))
        y0, y1 = max(0, ypx - ZOOM_HALF), min(height, ypx + ZOOM_HALF)
        x0, x1 = max(0, xpx - ZOOM_HALF), min(width, xpx + ZOOM_HALF)
        window = screen.to_pe(stack[cand["f0"]:cand["f0"] + cand["cp"], y0:y1, x0:x1].max(axis=0),
                              gain, offset, rqe, y0, y1, x0, x1)
        cand["zoom"] = zoom_crop(window, recentre=(dye != "ATTO565"))
    del stack

    # field of view, boxes, and the four traces inset (bottom right)
    fig, ax = plotter.one_column_plot(width=FOV_IN, height=FOV_IN)
    plotter.image_plot(ax, fov_img, cmap="gray", origin="upper", vmin=np.percentile(fov_img, 1),
                       vmax=np.percentile(fov_img, 99.5), scalebar=True, pixelsize=mf.PIXEL_SIZE_NM,
                       scalebarsize=10000, scalebarlabel="10 µm")
    for cand, colour in zip(top, BOX_COLOURS):
        ax.add_patch(mpatches.Rectangle((round(cand["xc"]) - HALF - 0.5, round(cand["yc"]) - HALF - 0.5),
                                        2 * HALF, 2 * HALF, linewidth=0.75, edgecolor=colour, facecolor="none"))
    trace_h = (INSET_H - (N_TRACES - 1) * GAP) / N_TRACES
    for i, (cand, colour) in enumerate(zip(top, BOX_COLOURS)):
        bottom = MARGIN + (N_TRACES - 1 - i) * (trace_h + GAP)
        ax_t = fig.add_axes([(FOV_IN - INSET_W - MARGIN) / FOV_IN, bottom / FOV_IN, INSET_W / FOV_IN, trace_h / FOV_IN])
        ax_t.set_facecolor((1, 1, 1, 0.75))
        for spine in ax_t.spines.values():
            spine.set_linewidth(0.4)
            spine.set_color(colour)
        ax_t.tick_params(labelsize=5, length=1.5, width=0.4, pad=1)
        trace = cand["trace"] - cand["trace"].min()
        t = np.arange(len(trace)) * exposure_s[dye]
        ax_t.plot(t, trace / 1e3, color="black", lw=0.75)
        ax_t.set_xlim(t[0], t[-1])
        ax_t.set_ylim(bottom=0)
        ax_t.set_ylabel(r"$10^3$ pe", fontsize=5, labelpad=2)
        if i == N_TRACES - 1:
            ax_t.set_xlabel("time / s", fontsize=6, labelpad=2)
        else:
            ax_t.set_xticklabels([])
    fig.savefig(FIG_DIR / f"{dye}_FOV_bleach.svg", dpi=600, format="svg", bbox_inches="tight")
    plt.show()

    # 2 x 2 PSF zoom-ins
    fig, axs = plotter.two_column_plot(nrows=2, ncols=2, width=2.0, height=2.0)
    for ax, cand, colour in zip(np.ravel(axs), top, BOX_COLOURS):
        plotter.image_plot(ax, cand["zoom"], cmap="gray", origin="upper", vmin=np.percentile(cand["zoom"], 1),
                           vmax=np.percentile(cand["zoom"], 99.9), scalebar=True, pixelsize=mf.PIXEL_SIZE_NM,
                           scalebarsize=500, scalebarlabel="500 nm")
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(colour)
    fig.savefig(FIG_DIR / f"{dye}_puncta_zoomin.svg", dpi=600, format="svg", bbox_inches="tight")
    plt.show()